# 05. Record Linkage y Consolidación

Este notebook implementa la lógica de vinculación de registros para identificar y fusionar duplicados entre las fuentes.

**Reglas de Negocio:**
1. **Bloqueo**: Por `email` y `phone_number`.
2. **Comparación**:
   - `email`: Exacto.
   - `phone_number`: Exacto.
   - `name`: Distancia de Levenshtein (umbral 0.85).
3. **Clasificación**: Un par es coincidencia si cumple **2 de las 3 condiciones**.
4. **Fusión**: Se conserva como registro maestro el del sistema interno.

In [6]:
import pandas as pd
import recordlinkage
import os
import warnings
warnings.filterwarnings('ignore')

# Configuración de rutas
PROCESSED_PATH = '../data/processed/'

# Carga del dataset unificado
df = pd.read_csv(os.path.join(PROCESSED_PATH, 'unified_dataset.csv'))

# Identificamos el origen (asumiendo que los primeros N registros son internos)
# En una implementación real, tendríamos una columna 'source'
# Para este proyecto, separaremos basándonos en el conocimiento del proceso previo
# Sin embargo, para Record Linkage trabajaremos sobre el mismo DataFrame buscando duplicados internos
print(f"Dataset cargado para consolidación: {df.shape}")

Dataset cargado para consolidación: (238780, 34)


## 1. Indexación (Bloqueo)
Creamos candidatos a comparación. Bloqueamos por `email` y `phone_number` para evitar comparaciones innecesarias.

In [7]:
indexer = recordlinkage.Index()
# Bloqueo por email (los nulos no generarán pares aquí automáticamente)
indexer.block('email')
# Bloqueo por teléfono para capturar los que no tienen email
indexer.block('phone_number')

candidate_links = indexer.index(df)
print(f"Pares candidatos generados: {len(candidate_links)}")

Pares candidatos generados: 134814


## 2. Comparación Probabilística
Definimos los criterios de similitud.

In [8]:
compare_cl = recordlinkage.Compare()

compare_cl.exact('email', 'email', label='email_match')
compare_cl.exact('phone_number', 'phone_number', label='phone_match')
compare_cl.string('name', 'name', method='levenshtein', threshold=0.85, label='name_match')

features = compare_cl.compute(candidate_links, df)
print("Resultados de comparación (primeras filas):")
print(features.head())

Resultados de comparación (primeras filas):
           email_match  phone_match  name_match
540  538             1            0         0.0
2034 1043            1            0         0.0
2309 1112            1            0         0.0
3864 2045            1            0         0.0
4343 2937            1            0         0.0


## 3. Clasificación y Fusión
Aplicamos la regla '2 de 3' y eliminamos los duplicados identificados.

In [9]:
# Sumamos los aciertos (cada match vale 1)
matches = features[features.sum(axis=1) >= 2]
print(f"Pares identificados para fusión: {len(matches)}")

# --- OPTIMIZACIÓN DE FUSIÓN (Evitando bucles lentos) ---
# Extraemos los orígenes de los índices involucrados
sources = df['source']

# Creamos un DataFrame con los pares de índices y sus fuentes
match_indices = matches.index.to_frame(index=False)
match_indices.columns = ['idx_a', 'idx_b']

match_indices['source_a'] = match_indices['idx_a'].map(sources)
match_indices['source_b'] = match_indices['idx_b'].map(sources)

# Identificamos registros de la fuente externa que tienen pareja en la interna
mask_b_is_ext = (match_indices['source_a'] == 'internal') & (match_indices['source_b'] == 'external')
mask_a_is_ext = (match_indices['source_a'] == 'external') & (match_indices['source_b'] == 'internal')

indices_b_fusionados = pd.concat([
    match_indices.loc[mask_b_is_ext, 'idx_b'],
    match_indices.loc[mask_a_is_ext, 'idx_a']
]).unique()

indices_a_validados = pd.concat([
    match_indices.loc[mask_b_is_ext, 'idx_a'],
    match_indices.loc[mask_a_is_ext, 'idx_b']
]).unique()

# 2. Crear Datasets Particionados
df_merged_only = df.loc[indices_a_validados].copy() # Registros de A que fueron validados por B
df_consolidated = df.drop(indices_b_fusionados).reset_index(drop=True) # Todo A + B huérfanos
df_orphans_b = df[(df['source'] == 'external') & (~df.index.isin(indices_b_fusionados))].copy()

print(f"Registros originales totales: {len(df)}")
print(f"Registros de Fuente B (externa) absorbidos por A: {len(indices_b_fusionados)}")
print(f"Registros 'huérfanos' de Fuente B: {len(df_orphans_b)}")
print(f"Dataset final consolidado: {len(df_consolidated)}")

Pares identificados para fusión: 116313
Registros originales totales: 238780
Registros de Fuente B (externa) absorbidos por A: 114924
Registros 'huérfanos' de Fuente B: 4466
Dataset final consolidado: 123856


## 4. Exportación
Guardamos el dataset definitivo para el análisis de negocio.

In [10]:
df_consolidated.to_csv(os.path.join(PROCESSED_PATH, 'consolidated_bookings.csv'), index=False)
df_merged_only.to_csv(os.path.join(PROCESSED_PATH, 'merged_records.csv'), index=False)
df_orphans_b.to_csv(os.path.join(PROCESSED_PATH, 'unmerged_external_records.csv'), index=False)

print(f"[RESULTADO] Archivos generados en {PROCESSED_PATH}:")
print("- consolidated_bookings.csv (Dataset Final)")
print("- merged_records.csv (Solo los que cruzaron correctamente)")
print("- unmerged_external_records.csv (Huérfanos de la fuente externa)")

[RESULTADO] Archivos generados en ../data/processed/:
- consolidated_bookings.csv (Dataset Final)
- merged_records.csv (Solo los que cruzaron correctamente)
- unmerged_external_records.csv (Huérfanos de la fuente externa)
